## 1. Environment Setup and Library Imports

In [ ]:
from __future__ import annotations
import re
import csv
import json
import math
import torch
import jieba
import joblib
import random
import hashlib
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from wordcloud import WordCloud
from dataclasses import dataclass
from datetime import date, datetime
from statistics import mean, median
from collections import Counter, defaultdict
from transformers import AutoTokenizer, AutoModel


warnings.filterwarnings("ignore", category=UserWarning)

## 2. File Path and Data Format Verification

In [ ]:
root = Path("./data")

paths = {
    "rumor_root": root / "CSDC-Rumor",
    "rumor_fact": root / "CSDC-Rumor" / "fact.json",
    "rumor_weibo_dir": root / "CSDC-Rumor" / "rumor_weibo",
    "rumor_fc_dir": root / "CSDC-Rumor" / "rumor_forward_comment",
    "news_root": root / "CSDC-News",
    "news_data_dir": root / "CSDC-News" / "data",
    "news_comment_dir": root / "CSDC-News" / "comment",
    "legal": root / "CSDC-Legal.json",
}

print("Root exists:", root.exists(), root.resolve())
for k, p in paths.items():
    print(f"{k:16} exists={p.exists()} path={p}")


def count_glob(dir_path: Path, pattern: str):
    if not dir_path.exists():
        return 0
    return sum(1 for _ in dir_path.glob(pattern))


print("\nFile counts:")
print("rumor_weibo *.json:", count_glob(paths["rumor_weibo_dir"], "*.json"))
print("rumor_fc    *.json:", count_glob(paths["rumor_fc_dir"], "*.json"))
print("news_data   *.txt :", count_glob(paths["news_data_dir"], "*.txt"))
print("news_comment*.txt :", count_glob(paths["news_comment_dir"], "*.txt"))

fact_path = paths["rumor_fact"]
if fact_path.exists():
    with fact_path.open("r", encoding="utf-8", errors="ignore") as f:
        first_nonempty = None
        for line in f:
            line = line.strip()
            if line:
                first_nonempty = line
                break

    print(
        "\nfact.json first non-empty line prefix:",
        (first_nonempty[:80] + "...") if first_nonempty else None,
    )

    try:
        obj = json.loads(first_nonempty) if first_nonempty else None
        print(
            "fact.json looks like JSONLines. First object keys:",
            sorted(list(obj.keys())) if isinstance(obj, dict) else type(obj),
        )
    except Exception as e:
        print("fact.json first-line json.loads failed:", repr(e))

## 3. Exploratory Analysis of fact.json in CSDC-Rumor

### 3.1 Explain Category Distribution, Top Tags, and Daily Volume Trend Analysis

In [ ]:
fact_path = Path("./data/CSDC-Rumor/fact.json")


def parse_date(s: str):
    try:
        return datetime.strptime(s.strip(), "%Y-%m-%d").date()
    except Exception:
        return None


explain_cnt = Counter()
tag_cnt = Counter()
daily_cnt = Counter()

n = 0
bad = 0
missing_date = 0

with fact_path.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except Exception:
            bad += 1
            continue

        n += 1
        explain = str(obj.get("explain", "")).strip()
        explain_cnt[explain] += 1

        tags = obj.get("tag") or []
        if isinstance(tags, list):
            tag_cnt.update([str(t).strip() for t in tags if str(t).strip()])

        d = parse_date(str(obj.get("date", "") or ""))
        if d is None:
            missing_date += 1
        else:
            daily_cnt[str(d)] += 1

print("fact lines parsed =", n, "bad_lines =", bad, "missing_date =", missing_date)

print("\n[Explain distribution]")
for k, v in explain_cnt.most_common():
    print(f"{k or '(EMPTY)'}\t{v}")

print("\n[Top10 tags]")
for k, v in tag_cnt.most_common(10):
    print(f"{k}\t{v}")

print("\n[Daily trend: first 15 days]")
for day in sorted(daily_cnt.keys())[:15]:
    print(day, daily_cnt[day])

print("\n[Daily trend: last 15 days]")
for day in sorted(daily_cnt.keys())[-15:]:
    print(day, daily_cnt[day])

### 3.2 Text Length Differences and Distinctive Keywords Across Explain Categories

In [ ]:
fact_path = Path("./data/CSDC-Rumor/fact.json")


def clean_text(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_tokenizer():
    try:

        def tok(x):
            return [w.strip() for w in jieba.lcut(x) if w.strip()]

        return tok
    except Exception:

        def tok(x):
            x = re.sub(r"\s+", "", x)
            if len(x) <= 1:
                return [x] if x else []
            return [x[i : i + 2] for i in range(len(x) - 1)]

        return tok


tokenize = build_tokenizer()

stopwords = {
    "",
    " ",
    "\n",
    "\t",
    "的",
    "了",
    "是",
    "我",
    "你",
    "他",
    "她",
    "它",
    "们",
    "在",
    "与",
    "和",
    "及",
    "或",
    "也",
    "都",
    "就",
    "而",
    "但",
    "一个",
    "我们",
    "你们",
    "他们",
    "这种",
    "这样",
    "那个",
    "这个",
    "：",
    "，",
    "。",
    "！",
    "？",
    "；",
    "、",
    "（",
    "）",
    "(",
    ")",
    "[",
    "]",
    "#",
    "@",
    "…",
    "-",
    "_",
    "/",
    "\\",
    "|",
    "《",
    "》",
    '"',
    "'",
}


def filter_tokens(tokens):
    out = []
    for t in tokens:
        t = t.strip()
        if not t:
            continue
        if t in stopwords:
            continue
        if len(t) <= 1:
            continue
        if re.fullmatch(r"[\W_]+", t):
            continue
        out.append(t)
    return out


records = []
with fact_path.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        d = clean_text(obj.get("date"))
        explain = clean_text(obj.get("explain"))
        title = clean_text(obj.get("title"))
        rumor = clean_text(obj.get("rumor"))
        abstract = clean_text(obj.get("abstract"))
        tags = obj.get("tag") or []
        if not isinstance(tags, list):
            tags = []
        tags = [clean_text(x) for x in tags if clean_text(x)]
        text = (title + " " + rumor + " " + abstract).strip()
        records.append(
            {
                "date": d,
                "explain": explain,
                "title": title,
                "rumor": rumor,
                "abstract": abstract,
                "tags": tags,
                "text": text,
            }
        )

by_explain = defaultdict(list)
for r in records:
    by_explain[r["explain"]].append(r)


def text_len_stats(items):
    lens = [len(x["text"]) for x in items]
    return {
        "count": len(lens),
        "mean_len": round(mean(lens), 2) if lens else 0.0,
        "median_len": round(median(lens), 2) if lens else 0.0,
        "min_len": min(lens) if lens else 0,
        "max_len": max(lens) if lens else 0,
    }


print("[Length stats by explain]")
for k in sorted(by_explain.keys(), key=lambda x: (-len(by_explain[x]), x)):
    s = text_len_stats(by_explain[k])
    print(k, s)

overall_tokens = []
group_tokens = {}
for k, items in by_explain.items():
    toks = []
    for x in items:
        toks.extend(filter_tokens(tokenize(x["text"])))
    group_tokens[k] = toks
    overall_tokens.extend(toks)

overall_cnt = Counter(overall_tokens)
overall_total = sum(overall_cnt.values()) or 1

print("\n[Top terms by explain]")
for k in sorted(group_tokens.keys(), key=lambda x: (-len(by_explain[x]), x)):
    c = Counter(group_tokens[k])
    top = c.most_common(15)
    print(k, top)

min_freq = 5
print("\n[Distinctive terms by explain]")
for k in sorted(group_tokens.keys(), key=lambda x: (-len(by_explain[x]), x)):
    c = Counter(group_tokens[k])
    group_total = sum(c.values()) or 1
    scored = []
    for term, f in c.items():
        if f < min_freq:
            continue
        p_g = f / group_total
        p_o = (overall_cnt[term] + 1) / (overall_total + len(overall_cnt))
        score = p_g / p_o
        scored.append((score, term, f))
    scored.sort(reverse=True)
    print(k, [(t, f, round(s, 2)) for s, t, f in scored[:15]])

print("\n[Examples per explain]")
for k in sorted(by_explain.keys(), key=lambda x: (-len(by_explain[x]), x)):
    items = by_explain[k]
    items_sorted = sorted(items, key=lambda x: len(x["text"]))
    short_ex = items_sorted[0]["title"]
    long_ex = items_sorted[-1]["title"]
    print(k, {"short_title": short_ex, "long_title": long_ex})

## 4. Exploratory Analysis of the Remaining CSDC-Rumor Folders

### 4.1 CSDC-Rumor Weibo Rumor Alignment and Interaction & Penalty Statistics

In [ ]:
root = Path("./data/CSDC-Rumor")
weibo_dir = root / "rumor_weibo"
fc_dir = root / "rumor_forward_comment"


def read_json(path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        return json.load(f)


def extract_code_from_filename(name):
    m = re.match(r"^\d{4}-\d{2}-\d{2}_(.+)\.json$", name)
    return m.group(1) if m else None


def extract_date_from_filename(name):
    m = re.match(r"^(?P<y>\d{4})-(?P<m>\d{2})-(?P<d>\d{2})_", name)
    if not m:
        return None
    return f"{m.group('y')}-{m.group('m')}-{m.group('d')}"


def extract_mute_days(result_text):
    if not result_text:
        return None
    s = str(result_text)
    m = re.search(r"禁言\s*(\d+)\s*天", s)
    if m:
        return int(m.group(1))
    return None


weibo = {}
mute_days_counter = Counter()

weibo_files = sorted(weibo_dir.glob("*.json"))
for p in weibo_files:
    obj = read_json(p)
    code = obj.get("rumorCode") or extract_code_from_filename(p.name) or ""
    publish_time = str(
        obj.get("publishTime") or ""
    ).strip() or extract_date_from_filename(p.name)
    visit_times = obj.get("visitTimes")
    if not isinstance(visit_times, int):
        visit_times = None
    title = str(obj.get("title") or "").strip()
    rumor_text = str(obj.get("rumorText") or "").strip()
    result = str(obj.get("result") or "").strip()
    md = extract_mute_days(result)
    if md is not None:
        mute_days_counter[md] += 1
    weibo[code] = {
        "code": code,
        "file": p.name,
        "publish_time": publish_time,
        "visit_times": visit_times,
        "title": title,
        "rumor_text": rumor_text,
        "result": result,
    }

fc_stats = defaultdict(lambda: {"comment": 0, "forward": 0})

fc_files = sorted(fc_dir.glob("*.json"))
for p in fc_files:
    code = extract_code_from_filename(p.name)
    if not code:
        continue
    arr = read_json(p)
    if not isinstance(arr, list):
        continue
    for item in arr:
        if not isinstance(item, dict):
            continue
        kind = str(item.get("comment_or_forward") or "").strip()
        if kind == "comment":
            fc_stats[code]["comment"] += 1
        elif kind == "forward":
            fc_stats[code]["forward"] += 1

aligned = []
for code, w in weibo.items():
    s = fc_stats.get(code, {"comment": 0, "forward": 0})
    total = int(s["comment"] + s["forward"])
    aligned.append(
        {
            "code": code,
            "publish_time": w["publish_time"],
            "visit_times": w["visit_times"],
            "comment_count": int(s["comment"]),
            "forward_count": int(s["forward"]),
            "total_interactions": total,
            "title": w["title"],
            "file": w["file"],
        }
    )

interaction_values = [x["total_interactions"] for x in aligned]
nonzero_values = [v for v in interaction_values if v > 0]


def stats(values):
    if not values:
        return {"count": 0, "min": 0, "median": 0, "mean": 0, "max": 0}
    return {
        "count": len(values),
        "min": min(values),
        "median": median(values),
        "mean": round(mean(values), 2),
        "max": max(values),
    }


print("weibo_count =", len(weibo))
print("forward_comment_file_count =", len(fc_files))
print(
    "weibo_with_interactions_count =",
    sum(1 for x in aligned if x["total_interactions"] > 0),
)

print("\n[Interactions stats: all weibos]")
print(stats(interaction_values))

print("\n[Interactions stats: nonzero only]")
print(stats(nonzero_values))

top10 = sorted(
    aligned,
    key=lambda x: (x["total_interactions"], x["visit_times"] or 0),
    reverse=True,
)[:10]
print("\n[Top10 by total_interactions]")
for i, x in enumerate(top10, 1):
    print(
        i,
        {
            "code": x["code"],
            "publish_time": x["publish_time"],
            "visit_times": x["visit_times"],
            "comment": x["comment_count"],
            "forward": x["forward_count"],
            "total": x["total_interactions"],
            "title": x["title"][:60],
            "file": x["file"],
        },
    )

print("\n[Mute days distribution top10]")
for k, v in mute_days_counter.most_common(10):
    print(k, v)

### 4.2 Top10 Rumor Weibo Sentiment & Anti-Rumor Analysis

In [ ]:
rumor_root = Path("./data/CSDC-Rumor")
weibo_dir = rumor_root / "rumor_weibo"
fc_dir = rumor_root / "rumor_forward_comment"


def read_json(path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        return json.load(f)


def clean_text(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def extract_code_from_filename(name):
    m = re.match(r"^\d{4}-\d{2}-\d{2}_(.+)\.json$", name)
    return m.group(1) if m else None


def build_tokenizer():
    try:

        def tokenize(text):
            return [w.strip() for w in jieba.lcut(text) if w.strip()]

        return tokenize
    except Exception:

        def tokenize(text):
            text = re.sub(r"\s+", "", text)
            if len(text) <= 1:
                return [text] if text else []
            return [text[i : i + 2] for i in range(len(text) - 1)]

        return tokenize


tokenize = build_tokenizer()

stopwords = {
    "\u7684",
    "\u4e86",
    "\u662f",
    "\u6211",
    "\u4f60",
    "\u4ed6",
    "\u5979",
    "\u5b83",
    "\u4eec",
    "\u5728",
    "\u4e0e",
    "\u548c",
    "\u53ca",
    "\u6216",
    "\u4e5f",
    "\u90fd",
    "\u5c31",
    "\u800c",
    "\u4f46",
    "\u4e00\u4e2a",
    "\u6211\u4eec",
    "\u4f60\u4eec",
    "\u4ed6\u4eec",
    "\u8fd9\u79cd",
    "\u8fd9\u6837",
    "\u90a3\u4e2a",
    "\u8fd9\u4e2a",
}


def filter_tokens(tokens):
    out = []
    for t in tokens:
        t = t.strip()
        if not t:
            continue
        if t in stopwords:
            continue
        if len(t) <= 1:
            continue
        if re.fullmatch(r"[\W_]+", t):
            continue
        out.append(t)
    return out


pos_words = {
    "\u611f\u8c22",
    "\u652f\u6301",
    "\u52a0\u6cb9",
    "\u70b9\u8d5e",
    "\u5e0c\u671b",
    "\u5e73\u5b89",
    "\u76f8\u4fe1",
    "\u4e0d\u6162",
    "\u4e0d\u9519",
    "\u8d5e",
    "\u597d",
    "\u771f\u7684",
    "\u771f\u76f8",
}
neg_words = {
    "\u8c23\u8a00",
    "\u9020\u8c23",
    "\u4e0d\u5b9e",
    "\u865a\u5047",
    "\u6050\u614c",
    "\u5bb3\u6015",
    "\u7126\u8651",
    "\u6124\u6012",
    "\u6076\u5fc3",
    "\u50bb\u903c",
    "\u50bb\u6279",
    "\u5bb3\u4eba",
    "\u5750\u7262",
    "\u5403\u7262\u996d",
    "\u8be5\u6b7b",
    "\u6709\u75c5",
    "\u62a5\u8b66",
    "\u4e25\u91cd",
}
negators = {"\u4e0d", "\u6ca1", "\u65e0", "\u522b", "\u975e", "\u672a"}

anti_terms = [
    "\u8c23\u8a00",
    "\u9020\u8c23",
    "\u4e0d\u5b9e",
    "\u865a\u5047",
    "\u8f9f\u8c23",
    "\u4e3e\u62a5",
    "\u7f51\u8b66",
    "\u5750\u7262",
    "\u5403\u7262\u996d",
    "\u7981\u8a00",
    "\u62a5\u8b66",
    "\u8fdd\u6cd5",
    "\u5904\u7f5a",
    "\u6cd5\u5f8b",
    "\u5220\u9664",
]


def sentiment_score(text):
    tokens = filter_tokens(tokenize(clean_text(text)))
    if not tokens:
        return 0.0
    pos = 0
    neg = 0
    for i, w in enumerate(tokens):
        if w in pos_words:
            if i > 0 and tokens[i - 1] in negators:
                neg += 1
            else:
                pos += 1
        elif w in neg_words:
            if i > 0 and tokens[i - 1] in negators:
                pos += 1
            else:
                neg += 1
    denom = pos + neg
    if denom == 0:
        return 0.0
    return (pos - neg) / denom


def sentiment_bucket(score, pos_th=0.2, neg_th=-0.2):
    if score >= pos_th:
        return "pos"
    if score <= neg_th:
        return "neg"
    return "neu"


def has_anti_term(text):
    t = clean_text(text)
    if not t:
        return False
    for k in anti_terms:
        if k in t:
            return True
    return False


weibo_meta = {}
for p in sorted(weibo_dir.glob("*.json")):
    obj = read_json(p)
    code = str(obj.get("rumorCode") or extract_code_from_filename(p.name) or "")
    weibo_meta[code] = {
        "code": code,
        "file": p.name,
        "publish_time": str(obj.get("publishTime") or "").strip(),
        "visit_times": obj.get("visitTimes")
        if isinstance(obj.get("visitTimes"), int)
        else None,
        "title": clean_text(obj.get("title")),
        "result": clean_text(obj.get("result")),
    }

counts = defaultdict(lambda: {"comment": 0, "forward": 0})
for p in sorted(fc_dir.glob("*.json")):
    code = extract_code_from_filename(p.name)
    if not code:
        continue
    arr = read_json(p)
    if not isinstance(arr, list):
        continue
    for item in arr:
        if not isinstance(item, dict):
            continue
        k = str(item.get("comment_or_forward") or "").strip()
        if k == "comment":
            counts[code]["comment"] += 1
        elif k == "forward":
            counts[code]["forward"] += 1

ranked = []
for code, meta in weibo_meta.items():
    c = counts.get(code, {"comment": 0, "forward": 0})
    total = int(c["comment"] + c["forward"])
    ranked.append(
        {
            "code": code,
            "title": meta.get("title", ""),
            "publish_time": meta.get("publish_time", ""),
            "visit_times": meta.get("visit_times", None),
            "comment": int(c["comment"]),
            "forward": int(c["forward"]),
            "total": total,
            "file": meta.get("file", ""),
        }
    )

top10 = sorted(ranked, key=lambda x: (x["total"], x["visit_times"] or 0), reverse=True)[
    :10
]
top_codes = set([x["code"] for x in top10])

texts_by_code = defaultdict(list)
texts_by_code_kind = defaultdict(lambda: {"comment": [], "forward": []})

for p in sorted(fc_dir.glob("*.json")):
    code = extract_code_from_filename(p.name)
    if code not in top_codes:
        continue
    arr = read_json(p)
    if not isinstance(arr, list):
        continue
    for item in arr:
        if not isinstance(item, dict):
            continue
        kind = str(item.get("comment_or_forward") or "").strip()
        text = clean_text(item.get("text"))
        if not text:
            continue
        texts_by_code[code].append((kind, text))
        if kind in ("comment", "forward"):
            texts_by_code_kind[code][kind].append(text)

all_items = []
for code in top_codes:
    all_items.extend(texts_by_code[code])


def summarize_items(items):
    if not items:
        return {"n": 0, "avg": 0.0, "pos": 0, "neu": 0, "neg": 0}
    scores = [sentiment_score(t) for _, t in items]
    buckets = [sentiment_bucket(s) for s in scores]
    c = Counter(buckets)
    return {
        "n": len(items),
        "avg": round(mean(scores), 4),
        "pos": int(c.get("pos", 0)),
        "neu": int(c.get("neu", 0)),
        "neg": int(c.get("neg", 0)),
    }


def summarize_texts(texts):
    if not texts:
        return {"n": 0, "avg": 0.0, "pos": 0, "neu": 0, "neg": 0}
    scores = [sentiment_score(t) for t in texts]
    buckets = [sentiment_bucket(s) for s in scores]
    c = Counter(buckets)
    return {
        "n": len(texts),
        "avg": round(mean(scores), 4),
        "pos": int(c.get("pos", 0)),
        "neu": int(c.get("neu", 0)),
        "neg": int(c.get("neg", 0)),
    }


all_summary = summarize_items(all_items)
print("[Top10 combined sentiment]")
print(all_summary)

comment_texts = []
forward_texts = []
for code in top_codes:
    comment_texts.extend(texts_by_code_kind[code]["comment"])
    forward_texts.extend(texts_by_code_kind[code]["forward"])

print("\n[Top10 comment sentiment]")
print(summarize_texts(comment_texts))

print("\n[Top10 forward sentiment]")
print(summarize_texts(forward_texts))

anti_counter = Counter()
anti_hits = 0
for _, text in all_items:
    hit = False
    for k in anti_terms:
        if k in text:
            anti_counter[k] += 1
            hit = True
    if hit:
        anti_hits += 1

print("\n[Top anti-rumor terms top20]")
print(anti_counter.most_common(20))
print("\n[Anti-rumor mention ratio]")
print(
    {
        "anti_texts": anti_hits,
        "all_texts": len(all_items),
        "ratio": round(anti_hits / max(1, len(all_items)), 4),
    }
)

print("\n[Per-rumor metrics for Top10]")
per_rumor_rows = []
for x in top10:
    code = x["code"]
    items = texts_by_code[code]
    if items:
        scores = [sentiment_score(t) for _, t in items]
        neg_ratio = sum(1 for s in scores if sentiment_bucket(s) == "neg") / len(scores)
        anti_ratio = sum(1 for _, t in items if has_anti_term(t)) / len(items)
        avg_s = mean(scores)
    else:
        neg_ratio = 0.0
        anti_ratio = 0.0
        avg_s = 0.0
    row = {
        "code": code,
        "total": x["total"],
        "comment": x["comment"],
        "forward": x["forward"],
        "avg_sentiment": round(avg_s, 4),
        "neg_ratio": round(neg_ratio, 4),
        "anti_ratio": round(anti_ratio, 4),
        "visit_times": x["visit_times"],
        "publish_time": x["publish_time"],
        "title": x["title"][:60],
    }
    per_rumor_rows.append(row)

for r in per_rumor_rows:
    print(r)

print("\n[Top15 comment terms per rumor]")
for x in top10:
    code = x["code"]
    toks = []
    for t in texts_by_code_kind[code]["comment"]:
        toks.extend(filter_tokens(tokenize(clean_text(t))))
    c = Counter(toks)
    print(code, c.most_common(15))

### 4.3 Daily Aggregation of Rumor Weibo Activity, Visits, and Mute Penalties

In [ ]:
data_dir = Path("./data/CSDC-Rumor/rumor_weibo")
out_dir = Path("./outputs/step5")
out_dir.mkdir(parents=True, exist_ok=True)


def read_json(path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        return json.load(f)


def parse_date(s):
    if s is None:
        return None
    s = str(s).strip()
    if not s:
        return None
    try:
        return datetime.strptime(s, "%Y-%m-%d").date()
    except Exception:
        m = re.search(r"(\d{4}-\d{2}-\d{2})", s)
        if m:
            try:
                return datetime.strptime(m.group(1), "%Y-%m-%d").date()
            except Exception:
                return None
        return None


def date_from_filename(name):
    m = re.match(r"^(?P<y>\d{4})-(?P<m>\d{2})-(?P<d>\d{2})_", name)
    if not m:
        return None
    try:
        return datetime.strptime(
            f"{m.group('y')}-{m.group('m')}-{m.group('d')}", "%Y-%m-%d"
        ).date()
    except Exception:
        return None


def extract_mute_days(result_text):
    if result_text is None:
        return None
    s = str(result_text)
    m = re.search(r"\u7981\u8a00\s*(\d+)\s*\u5929", s)
    if m:
        try:
            return int(m.group(1))
        except Exception:
            return None
    return None


daily_weibo_count = Counter()
daily_visit_values = defaultdict(list)
daily_mute_days = defaultdict(list)
mute_days_dist = Counter()

files = sorted(data_dir.glob("*.json"))
for p in files:
    obj = read_json(p)
    d = parse_date(obj.get("publishTime")) or date_from_filename(p.name)
    if d is None:
        continue

    daily_weibo_count[str(d)] += 1

    vt = obj.get("visitTimes")
    if isinstance(vt, int):
        daily_visit_values[str(d)].append(vt)

    md = extract_mute_days(obj.get("result"))
    if md is not None:
        daily_mute_days[str(d)].append(md)
        mute_days_dist[md] += 1

all_days = sorted(daily_weibo_count.keys())
print("files_scanned =", len(files))
print("days_covered =", len(all_days))
if all_days:
    print("date_range =", {"start": all_days[0], "end": all_days[-1]})

top_days = sorted(all_days, key=lambda x: daily_weibo_count[x], reverse=True)[:10]
print("\nTop10 days by weibo_count:")
for day in top_days:
    print(day, daily_weibo_count[day])

print("\nTop10 mute_days distribution:")
for k, v in mute_days_dist.most_common(10):
    print(k, v)

rows = []
for day in all_days:
    visits = daily_visit_values.get(day, [])
    mds = daily_mute_days.get(day, [])
    rows.append(
        {
            "date": day,
            "weibo_count": int(daily_weibo_count[day]),
            "avg_visit_times": round(mean(visits), 2) if visits else "",
            "mute_cases": len(mds),
            "avg_mute_days": round(mean(mds), 2) if mds else "",
        }
    )

out_path = out_dir / "weibo_daily.csv"
with out_path.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(
        f,
        fieldnames=[
            "date",
            "weibo_count",
            "avg_visit_times",
            "mute_cases",
            "avg_mute_days",
        ],
    )
    w.writeheader()
    for r in rows:
        w.writerow(r)

print("\nwritten =", str(out_path))

### 4.4 Daily Weibo Interactions, Sentiment, and Anti-Rumor Trends Merged

In [ ]:
data_dir = Path("./data/CSDC-Rumor/rumor_forward_comment")
weibo_daily_path = Path("./outputs/step5/weibo_daily.csv")
out_path = Path("./outputs/step5/rumor_daily_merged.csv")


def read_json(path):
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        return json.load(f)


def parse_month_day(s, year):
    if s is None:
        return None
    s = str(s).strip()
    m = re.search(r"(\d{1,2})\u6708(\d{1,2})\u65e5", s)
    if not m:
        return None
    mm = int(m.group(1))
    dd = int(m.group(2))
    try:
        return date(year, mm, dd)
    except Exception:
        return None


def year_from_filename(name, default_year=2020):
    m = re.match(r"^(?P<y>\d{4})-\d{2}-\d{2}_", name)
    if m:
        return int(m.group("y"))
    return default_year


def build_tokenizer():
    try:
        import jieba

        def tokenize(text):
            return [w.strip() for w in jieba.lcut(text) if w.strip()]

        return tokenize
    except Exception:

        def tokenize(text):
            text = re.sub(r"\s+", "", str(text))
            if len(text) <= 1:
                return [text] if text else []
            return [text[i : i + 2] for i in range(len(text) - 1)]

        return tokenize


tokenize = build_tokenizer()

stopwords = {
    "\u7684",
    "\u4e86",
    "\u662f",
    "\u6211",
    "\u4f60",
    "\u4ed6",
    "\u5979",
    "\u5b83",
    "\u4eec",
    "\u5728",
    "\u4e0e",
    "\u548c",
    "\u53ca",
    "\u6216",
    "\u4e5f",
    "\u90fd",
    "\u5c31",
    "\u800c",
    "\u4f46",
    "\u4e00\u4e2a",
    "\u6211\u4eec",
    "\u4f60\u4eec",
    "\u4ed6\u4eec",
    "\u8fd9\u79cd",
    "\u8fd9\u6837",
    "\u90a3\u4e2a",
    "\u8fd9\u4e2a",
}


def filter_tokens(tokens):
    out = []
    for t in tokens:
        t = str(t).strip()
        if not t:
            continue
        if t in stopwords:
            continue
        if len(t) <= 1:
            continue
        if re.fullmatch(r"[\W_]+", t):
            continue
        out.append(t)
    return out


pos_words = {
    "\u611f\u8c22",
    "\u652f\u6301",
    "\u52a0\u6cb9",
    "\u70b9\u8d5e",
    "\u5e0c\u671b",
    "\u5e73\u5b89",
    "\u76f8\u4fe1",
    "\u8d5e",
    "\u597d",
    "\u771f\u7684",
    "\u771f\u76f8",
}
neg_words = {
    "\u8c23\u8a00",
    "\u9020\u8c23",
    "\u4e0d\u5b9e",
    "\u865a\u5047",
    "\u6050\u614c",
    "\u5bb3\u6015",
    "\u7126\u8651",
    "\u6124\u6012",
    "\u6076\u5fc3",
    "\u50bb\u903c",
    "\u50bb\u6279",
    "\u5bb3\u4eba",
    "\u5750\u7262",
    "\u5403\u7262\u996d",
    "\u6709\u75c5",
    "\u62a5\u8b66",
    "\u4e25\u91cd",
}
negators = {"\u4e0d", "\u6ca1", "\u65e0", "\u522b", "\u975e", "\u672a"}

anti_terms = [
    "\u8f9f\u8c23",
    "\u9020\u8c23",
    "\u4e0d\u5b9e",
    "\u8c23\u8a00",
    "\u4e3e\u62a5",
    "\u7f51\u8b66",
    "\u7981\u8a00",
    "\u5750\u7262",
    "\u5403\u7262\u996d",
    "\u62a5\u8b66",
    "\u6cd5\u5f8b",
    "\u5904\u7f5a",
    "\u5220\u9664",
    "\u8fdd\u6cd5",
    "\u865a\u5047",
]


def sentiment_score(text):
    tokens = filter_tokens(tokenize(text))
    if not tokens:
        return 0.0
    pos = 0
    neg = 0
    for i, w in enumerate(tokens):
        if w in pos_words:
            if i > 0 and tokens[i - 1] in negators:
                neg += 1
            else:
                pos += 1
        elif w in neg_words:
            if i > 0 and tokens[i - 1] in negators:
                pos += 1
            else:
                neg += 1
    denom = pos + neg
    if denom == 0:
        return 0.0
    return (pos - neg) / denom


daily = defaultdict(
    lambda: {
        "comment_count": 0,
        "forward_count": 0,
        "scores_all": [],
        "scores_comment": [],
        "scores_forward": [],
        "anti_hits": 0,
        "text_count": 0,
    }
)

files = sorted(data_dir.glob("*.json"))
for p in files:
    year = year_from_filename(p.name, 2020)
    arr = read_json(p)
    if not isinstance(arr, list):
        continue
    for item in arr:
        if not isinstance(item, dict):
            continue
        kind = str(item.get("comment_or_forward") or "").strip()
        text = str(item.get("text") or "").strip()
        if not text:
            continue
        d = parse_month_day(item.get("date"), year)
        if d is None:
            continue
        day = str(d)
        daily[day]["text_count"] += 1
        s = sentiment_score(text)
        daily[day]["scores_all"].append(s)
        if kind == "comment":
            daily[day]["comment_count"] += 1
            daily[day]["scores_comment"].append(s)
        elif kind == "forward":
            daily[day]["forward_count"] += 1
            daily[day]["scores_forward"].append(s)
        hit = any(k in text for k in anti_terms)
        if hit:
            daily[day]["anti_hits"] += 1

daily_rows = {}
for day in daily.keys():
    a = daily[day]["scores_all"]
    c = daily[day]["scores_comment"]
    f = daily[day]["scores_forward"]
    total = daily[day]["comment_count"] + daily[day]["forward_count"]
    daily_rows[day] = {
        "date": day,
        "comment_count": int(daily[day]["comment_count"]),
        "forward_count": int(daily[day]["forward_count"]),
        "interaction_total": int(total),
        "avg_sentiment_all": round(mean(a), 4) if a else "",
        "avg_sentiment_comment": round(mean(c), 4) if c else "",
        "avg_sentiment_forward": round(mean(f), 4) if f else "",
        "anti_ratio": round(
            daily[day]["anti_hits"] / max(1, daily[day]["text_count"]), 4
        ),
    }

weibo_rows = {}
with weibo_daily_path.open("r", encoding="utf-8", errors="ignore") as f:
    r = csv.DictReader(f)
    for row in r:
        weibo_rows[row["date"]] = row

all_days = sorted(set(weibo_rows.keys()) | set(daily_rows.keys()))
merged = []
for day in all_days:
    w = weibo_rows.get(day, {})
    d = daily_rows.get(day, {})
    merged.append(
        {
            "date": day,
            "weibo_count": w.get("weibo_count", "0"),
            "avg_visit_times": w.get("avg_visit_times", ""),
            "mute_cases": w.get("mute_cases", "0"),
            "avg_mute_days": w.get("avg_mute_days", ""),
            "comment_count": d.get("comment_count", 0),
            "forward_count": d.get("forward_count", 0),
            "interaction_total": d.get("interaction_total", 0),
            "avg_sentiment_all": d.get("avg_sentiment_all", ""),
            "avg_sentiment_comment": d.get("avg_sentiment_comment", ""),
            "avg_sentiment_forward": d.get("avg_sentiment_forward", ""),
            "anti_ratio": d.get("anti_ratio", 0.0),
        }
    )

with out_path.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(
        f,
        fieldnames=[
            "date",
            "weibo_count",
            "avg_visit_times",
            "mute_cases",
            "avg_mute_days",
            "comment_count",
            "forward_count",
            "interaction_total",
            "avg_sentiment_all",
            "avg_sentiment_comment",
            "avg_sentiment_forward",
            "anti_ratio",
        ],
    )
    w.writeheader()
    for row in merged:
        w.writerow(row)

top_interaction_days = sorted(
    [x for x in merged if int(x["comment_count"]) + int(x["forward_count"]) > 0],
    key=lambda x: int(x["interaction_total"]),
    reverse=True,
)[:10]

print("fc_files_scanned =", len(files))
print("merged_days =", len(merged))
if merged:
    print(
        "merged_date_range =", {"start": merged[0]["date"], "end": merged[-1]["date"]}
    )

print("\nTop10 days by interaction_total:")
for x in top_interaction_days:
    print(
        x["date"],
        x["interaction_total"],
        {
            "weibo_count": x["weibo_count"],
            "anti_ratio": x["anti_ratio"],
            "avg_sentiment_all": x["avg_sentiment_all"],
        },
    )

print("\nwritten =", str(out_path))

### 4.5 Daily Trends of Rumor Posts, Interactions, and Anti-Rumor Discourse

In [ ]:
in_path = Path("./outputs/step5/rumor_daily_merged.csv")
out_dir = Path("./outputs/step6")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "rumor_trends.png"


def to_int(x):
    try:
        return int(str(x).strip())
    except Exception:
        return 0


def to_float(x):
    try:
        s = str(x).strip()
        return float(s) if s else 0.0
    except Exception:
        return 0.0


def to_date(x):
    return datetime.strptime(str(x).strip(), "%Y-%m-%d")


def rolling_mean(values, window):
    out = []
    for i in range(len(values)):
        left = max(0, i - window + 1)
        chunk = values[left : i + 1]
        out.append(mean(chunk) if chunk else 0.0)
    return out


try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

rows = []
with in_path.open("r", encoding="utf-8", errors="ignore") as f:
    r = csv.DictReader(f)
    for row in r:
        d = to_date(row["date"])
        rows.append(
            {
                "date": d,
                "weibo_count": to_int(row.get("weibo_count")),
                "interaction_total": to_int(row.get("interaction_total")),
                "anti_ratio": to_float(row.get("anti_ratio")),
            }
        )

rows.sort(key=lambda x: x["date"])

x = [r["date"] for r in rows]
y_weibo = [r["weibo_count"] for r in rows]
y_inter = [r["interaction_total"] for r in rows]
y_anti = [r["anti_ratio"] for r in rows]

w = 7
y_weibo_ma = rolling_mean(y_weibo, w)
y_inter_ma = rolling_mean(y_inter, w)
y_anti_ma = rolling_mean(y_anti, w)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True, constrained_layout=True)

c1 = "#1f77b4"
c2 = "#d62728"
c3 = "#2ca02c"

axes[0].plot(x, y_weibo, color=c1, linewidth=1.6, alpha=0.45)
axes[0].plot(x, y_weibo_ma, color=c1, linewidth=2.6)
axes[0].set_ylabel("weibo_count")
axes[0].set_title("Daily Rumor Weibo Volume")
axes[0].grid(True, alpha=0.25)

axes[1].plot(x, y_inter, color=c2, linewidth=1.6, alpha=0.35)
axes[1].plot(x, y_inter_ma, color=c2, linewidth=2.6)
axes[1].set_ylabel("interaction_total")
axes[1].set_title("Daily Interaction Volume (symlog scale)")
axes[1].set_yscale("symlog", linthresh=50)
axes[1].grid(True, alpha=0.25)

axes[2].plot(x, y_anti, color=c3, linewidth=1.6, alpha=0.45)
axes[2].plot(x, y_anti_ma, color=c3, linewidth=2.6)
axes[2].set_ylabel("anti_ratio")
axes[2].set_title("Daily Anti-rumor Mention Ratio")
axes[2].set_ylim(bottom=0.0)
axes[2].grid(True, alpha=0.25)

locator = mdates.AutoDateLocator(minticks=6, maxticks=10)
formatter = mdates.ConciseDateFormatter(locator)
axes[2].xaxis.set_major_locator(locator)
axes[2].xaxis.set_major_formatter(formatter)

fig.suptitle("Rumor Dynamics Over Time (Raw vs 7-day Moving Average)", fontsize=14)
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print("points =", len(rows))
print("written =", str(out_path))

## 5. Word Clouds for Rumor Topics and User Interactions

In [ ]:
def find_font_path():
    local_font = Path.cwd() / "SimHei.ttf"
    if local_font.exists():
        return str(local_font)

    candidates = [
        "/System/Library/Fonts/PingFang.ttc",
        "/System/Library/Fonts/STHeiti Medium.ttc",
        "/System/Library/Fonts/STHeiti Light.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/System/Library/Fonts/Supplemental/Songti.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        "/Library/Fonts/Microsoft YaHei.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "C:\\Windows\\Fonts\\msyh.ttc",
        "C:\\Windows\\Fonts\\msyh.ttf",
        "C:\\Windows\\Fonts\\simhei.ttf",
        "C:\\Windows\\Fonts\\simsun.ttc",
    ]
    for p in candidates:
        if Path(p).exists():
            return p
    return None


def clean_text(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_tokenizer():
    try:
        import jieba

        def tokenize(text):
            return [w.strip() for w in jieba.lcut(text) if w.strip()]

        return tokenize
    except Exception:

        def tokenize(text):
            text = re.sub(r"\s+", "", str(text))
            if len(text) <= 1:
                return [text] if text else []
            return [text[i : i + 2] for i in range(len(text) - 1)]

        return tokenize


def build_stopwords():
    items = [
        "\u7684",
        "\u4e86",
        "\u662f",
        "\u6211",
        "\u4f60",
        "\u4ed6",
        "\u5979",
        "\u5b83",
        "\u4eec",
        "\u5728",
        "\u4e0e",
        "\u548c",
        "\u53ca",
        "\u6216",
        "\u4e5f",
        "\u90fd",
        "\u5c31",
        "\u800c",
        "\u4f46",
        "\u4e00\u4e2a",
        "\u6211\u4eec",
        "\u4f60\u4eec",
        "\u4ed6\u4eec",
        "\u8fd9\u79cd",
        "\u8fd9\u6837",
        "\u90a3\u4e2a",
        "\u8fd9\u4e2a",
        "\u8fd8",
        "\u5f88",
        "\u8fd9\u4e9b",
        "\u90a3\u4e9b",
        "\u6ca1\u6709",
        "\u56e0\u4e3a",
        "\u6240\u4ee5",
    ]
    return set(items)


def filter_tokens(tokens, stopwords):
    out = []
    for t in tokens:
        t = str(t).strip()
        if not t:
            continue
        if t in stopwords:
            continue
        if len(t) <= 1:
            continue
        if re.fullmatch(r"[\W_]+", t):
            continue
        out.append(t)
    return out


def generate_wordcloud(
    freq, out_path, font_path, width=1600, height=1000, max_words=200
):
    if not font_path:
        raise RuntimeError(
            "no font found for CJK rendering. set a valid font_path in find_font_path()"
        )
    wc = WordCloud(
        width=width,
        height=height,
        background_color="white",
        font_path=font_path,
        max_words=max_words,
        collocations=False,
        prefer_horizontal=0.9,
    )
    wc.generate_from_frequencies(dict(freq))
    wc.to_file(str(out_path))


out_dir = Path("./outputs/step6")
out_dir.mkdir(parents=True, exist_ok=True)

tokenize = build_tokenizer()
stopwords = build_stopwords()
font_path = find_font_path()

fact_path = Path("./data/CSDC-Rumor/fact.json")
fact_counter = Counter()

with fact_path.open("r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = " ".join(
            [
                clean_text(obj.get("title")),
                clean_text(obj.get("rumor")),
                clean_text(obj.get("abstract")),
            ]
        ).strip()
        if not text:
            continue
        toks = filter_tokens(tokenize(text), stopwords)
        fact_counter.update(toks)

fact_out = out_dir / "wordcloud_fact.png"
generate_wordcloud(fact_counter, fact_out, font_path)
print("written =", str(fact_out), "unique_terms =", len(fact_counter))

fc_dir = Path("./data/CSDC-Rumor/rumor_forward_comment")
all_counter = Counter()
comment_counter = Counter()
forward_counter = Counter()

fc_files = sorted(fc_dir.glob("*.json"))
for p in fc_files:
    arr = json.loads(p.read_text(encoding="utf-8", errors="ignore"))
    if not isinstance(arr, list):
        continue
    for item in arr:
        if not isinstance(item, dict):
            continue
        kind = str(item.get("comment_or_forward") or "").strip()
        text = clean_text(item.get("text"))
        if not text:
            continue
        toks = filter_tokens(tokenize(text), stopwords)
        if not toks:
            continue
        all_counter.update(toks)
        if kind == "comment":
            comment_counter.update(toks)
        elif kind == "forward":
            forward_counter.update(toks)

all_out = out_dir / "wordcloud_interactions_all.png"
comment_out = out_dir / "wordcloud_interactions_comment.png"
forward_out = out_dir / "wordcloud_interactions_forward.png"

generate_wordcloud(all_counter, all_out, font_path)
generate_wordcloud(comment_counter, comment_out, font_path)
generate_wordcloud(forward_counter, forward_out, font_path)

print("written =", str(all_out), "unique_terms =", len(all_counter))
print("written =", str(comment_out), "unique_terms =", len(comment_counter))
print("written =", str(forward_out), "unique_terms =", len(forward_counter))
print("fc_files_scanned =", len(fc_files))

## 6. Peak-Annotated Trend Analysis with Lagged Correlations and Peak-Day Export

In [ ]:
@dataclass(frozen=True)
class DailyPoint:
    dt: datetime
    weibo_count: int
    interaction_total: int
    anti_ratio: float


class RumorDailyLoader:
    def __init__(self, csv_path: Path):
        self._csv_path = csv_path

    def load(self) -> list[DailyPoint]:
        rows: list[DailyPoint] = []
        with self._csv_path.open("r", encoding="utf-8", errors="ignore") as f:
            r = csv.DictReader(f)
            for row in r:
                dt = datetime.strptime(str(row["date"]).strip(), "%Y-%m-%d")
                rows.append(
                    DailyPoint(
                        dt=dt,
                        weibo_count=self._to_int(row.get("weibo_count")),
                        interaction_total=self._to_int(row.get("interaction_total")),
                        anti_ratio=self._to_float(row.get("anti_ratio")),
                    )
                )
        rows.sort(key=lambda x: x.dt)
        return rows

    def _to_int(self, x) -> int:
        try:
            return int(str(x).strip())
        except Exception:
            return 0

    def _to_float(self, x) -> float:
        try:
            s = str(x).strip()
            return float(s) if s else 0.0
        except Exception:
            return 0.0


class TrendAnalyzer:
    def pearson(self, a: list[float], b: list[float]) -> float:
        if len(a) != len(b) or len(a) < 2:
            return float("nan")
        ma = mean(a)
        mb = mean(b)
        num = 0.0
        da = 0.0
        db = 0.0
        for i in range(len(a)):
            xa = a[i] - ma
            xb = b[i] - mb
            num += xa * xb
            da += xa * xa
            db += xb * xb
        den = math.sqrt(da) * math.sqrt(db)
        if den == 0.0:
            return float("nan")
        return num / den

    def lagged_pearson(
        self, a: list[float], b: list[float], max_lag: int
    ) -> list[dict]:
        out: list[dict] = []
        for lag in range(-max_lag, max_lag + 1):
            if lag == 0:
                aa = a
                bb = b
            elif lag > 0:
                aa = a[:-lag]
                bb = b[lag:]
            else:
                aa = a[-lag:]
                bb = b[:lag]

            corr = self.pearson(aa, bb) if len(aa) >= 2 else float("nan")
            out.append({"lag": lag, "corr": corr, "n": len(aa)})
        return out

    def best_lag(self, rows: list[dict]) -> dict:
        valid = []
        for r in rows:
            c = r.get("corr")
            if isinstance(c, float) and not math.isnan(c):
                valid.append(r)
        if not valid:
            return {"lag": None, "corr": float("nan"), "n": 0}
        valid.sort(key=lambda x: abs(x["corr"]), reverse=True)
        return valid[0]


class TrendPlotter:
    def __init__(self, dpi: int = 300):
        self._dpi = int(dpi)

    def plot(
        self,
        points: list[DailyPoint],
        out_path: Path,
        top_k: int = 3,
        ma_window: int = 7,
    ) -> None:
        x = [p.dt for p in points]
        y_weibo = [float(p.weibo_count) for p in points]
        y_inter = [float(p.interaction_total) for p in points]
        y_anti = [float(p.anti_ratio) for p in points]

        y_weibo_ma = self._rolling_mean(y_weibo, ma_window)
        y_inter_ma = self._rolling_mean(y_inter, ma_window)
        y_anti_ma = self._rolling_mean(y_anti, ma_window)

        try:
            plt.style.use("seaborn-v0_8-whitegrid")
        except Exception:
            pass

        fig, axes = plt.subplots(
            3, 1, figsize=(14, 9), sharex=True, constrained_layout=True
        )

        c1 = "#1f77b4"
        c2 = "#d62728"
        c3 = "#2ca02c"

        axes[0].plot(x, y_weibo, color=c1, linewidth=1.6, alpha=0.45)
        axes[0].plot(x, y_weibo_ma, color=c1, linewidth=2.6)
        axes[0].set_ylabel("weibo_count")
        axes[0].set_title("Daily Rumor Weibo Volume")
        axes[0].grid(True, alpha=0.25)

        axes[1].plot(x, y_inter, color=c2, linewidth=1.6, alpha=0.35)
        axes[1].plot(x, y_inter_ma, color=c2, linewidth=2.6)
        axes[1].set_ylabel("interaction_total")
        axes[1].set_title("Daily Interaction Volume (symlog scale)")
        axes[1].set_yscale("symlog", linthresh=50)
        axes[1].grid(True, alpha=0.25)

        axes[2].plot(x, y_anti, color=c3, linewidth=1.6, alpha=0.45)
        axes[2].plot(x, y_anti_ma, color=c3, linewidth=2.6)
        axes[2].set_ylabel("anti_ratio")
        axes[2].set_title("Daily Anti-rumor Mention Ratio")
        axes[2].set_ylim(bottom=0.0)
        axes[2].grid(True, alpha=0.25)

        locator = mdates.AutoDateLocator(minticks=6, maxticks=10)
        formatter = mdates.ConciseDateFormatter(locator)
        axes[2].xaxis.set_major_locator(locator)
        axes[2].xaxis.set_major_formatter(formatter)

        self._annotate_topk(axes[0], x, y_weibo, top_k, color=c1)
        self._annotate_topk(axes[1], x, y_inter, top_k, color=c2)
        self._annotate_topk(axes[2], x, y_anti, top_k, color=c3)

        fig.suptitle(
            f"Rumor Dynamics Over Time (Raw vs {ma_window}-day Moving Average)",
            fontsize=14,
        )
        fig.savefig(out_path, dpi=self._dpi, bbox_inches="tight")
        plt.close(fig)

    def _rolling_mean(self, values: list[float], window: int) -> list[float]:
        w = max(1, int(window))
        out: list[float] = []
        for i in range(len(values)):
            left = max(0, i - w + 1)
            chunk = values[left : i + 1]
            out.append(mean(chunk) if chunk else 0.0)
        return out

    def _topk_indices(self, ys: list[float], k: int) -> list[int]:
        kk = max(0, int(k))
        items = list(range(len(ys)))
        items.sort(key=lambda i: ys[i], reverse=True)
        return items[:kk]

    def _annotate_topk(
        self, ax, xs: list[datetime], ys: list[float], k: int, color: str
    ) -> None:
        idxs = self._topk_indices(ys, k)
        offsets = [(18, 18), (18, -24), (-42, 18), (-42, -24), (30, 0), (-60, 0)]
        for j, idx in enumerate(idxs):
            dt = xs[idx]
            val = ys[idx]
            dx, dy = offsets[j % len(offsets)]
            label = f"{dt.strftime('%Y-%m-%d')} ({self._fmt(val)})"
            ax.scatter([dt], [val], s=36, color=color, zorder=5)
            ax.annotate(
                label,
                xy=(dt, val),
                xytext=(dx, dy),
                textcoords="offset points",
                fontsize=9,
                color=color,
                arrowprops=dict(arrowstyle="->", color=color, alpha=0.7, linewidth=1.0),
            )

    def _fmt(self, v: float) -> str:
        if abs(v) >= 1000:
            return f"{v:.0f}"
        if abs(v) >= 10:
            return f"{v:.1f}"
        if abs(v) >= 1:
            return f"{v:.2f}"
        return f"{v:.4f}"


class PeakExporter:
    def export(self, points: list[DailyPoint], out_path: Path, top_k: int) -> None:
        weibo = [(p.dt, p.weibo_count) for p in points]
        inter = [(p.dt, p.interaction_total) for p in points]
        anti = [(p.dt, p.anti_ratio) for p in points]

        weibo.sort(key=lambda x: x[1], reverse=True)
        inter.sort(key=lambda x: x[1], reverse=True)
        anti.sort(key=lambda x: x[1], reverse=True)

        with out_path.open("w", encoding="utf-8", newline="") as f:
            w = csv.writer(f)
            w.writerow(["metric", "rank", "date", "value"])

            for i in range(max(0, int(top_k))):
                if i < len(weibo):
                    w.writerow(
                        [
                            "weibo_count",
                            i + 1,
                            weibo[i][0].strftime("%Y-%m-%d"),
                            weibo[i][1],
                        ]
                    )
                if i < len(inter):
                    w.writerow(
                        [
                            "interaction_total",
                            i + 1,
                            inter[i][0].strftime("%Y-%m-%d"),
                            inter[i][1],
                        ]
                    )
                if i < len(anti):
                    w.writerow(
                        [
                            "anti_ratio",
                            i + 1,
                            anti[i][0].strftime("%Y-%m-%d"),
                            f"{anti[i][1]:.6f}",
                        ]
                    )


class LagExporter:
    def export(self, out_path: Path, rows: list[dict]) -> None:
        with out_path.open("w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["pair", "lag", "corr", "n"])
            w.writeheader()
            for r in rows:
                w.writerow(
                    {
                        "pair": r.get("pair", ""),
                        "lag": r.get("lag", ""),
                        "corr": r.get("corr", ""),
                        "n": r.get("n", ""),
                    }
                )


class App:
    def __init__(self, in_csv: Path, out_dir: Path):
        self._in_csv = in_csv
        self._out_dir = out_dir

        self._loader = RumorDailyLoader(in_csv)
        self._analyzer = TrendAnalyzer()
        self._plotter = TrendPlotter(dpi=300)
        self._peak_exporter = PeakExporter()
        self._lag_exporter = LagExporter()

    def run(self) -> None:
        self._out_dir.mkdir(parents=True, exist_ok=True)

        points = self._loader.load()
        if not points:
            raise RuntimeError("no daily points loaded")

        plot_path = self._out_dir / "rumor_trends_annotated.png"
        self._plotter.plot(points, plot_path, top_k=3, ma_window=7)

        peak_path = self._out_dir / "peak_days.csv"
        self._peak_exporter.export(points, peak_path, top_k=5)

        weibo_vals = [float(p.weibo_count) for p in points]
        inter_vals = [float(p.interaction_total) for p in points]
        anti_vals = [float(p.anti_ratio) for p in points]

        corr_wi = self._analyzer.pearson(weibo_vals, inter_vals)
        corr_wa = self._analyzer.pearson(weibo_vals, anti_vals)
        corr_ia = self._analyzer.pearson(inter_vals, anti_vals)

        print("points =", len(points))
        print("written =", str(plot_path))
        print("written =", str(peak_path))
        print("pearson_same_day weibo_vs_interaction =", f"{corr_wi:.6f}")
        print("pearson_same_day weibo_vs_anti_ratio  =", f"{corr_wa:.6f}")
        print("pearson_same_day inter_vs_anti_ratio  =", f"{corr_ia:.6f}")

        lag_rows: list[dict] = []

        l_wi = self._analyzer.lagged_pearson(weibo_vals, inter_vals, max_lag=7)
        for r in l_wi:
            r["pair"] = "weibo_vs_interaction"
        lag_rows.extend(l_wi)
        best_wi = self._analyzer.best_lag(l_wi)

        l_wa = self._analyzer.lagged_pearson(weibo_vals, anti_vals, max_lag=7)
        for r in l_wa:
            r["pair"] = "weibo_vs_anti_ratio"
        lag_rows.extend(l_wa)
        best_wa = self._analyzer.best_lag(l_wa)

        l_ia = self._analyzer.lagged_pearson(inter_vals, anti_vals, max_lag=7)
        for r in l_ia:
            r["pair"] = "interaction_vs_anti_ratio"
        lag_rows.extend(l_ia)
        best_ia = self._analyzer.best_lag(l_ia)

        lag_path = self._out_dir / "lag_correlations.csv"
        self._lag_exporter.export(lag_path, lag_rows)
        print("written =", str(lag_path))

        print(
            "best_lag weibo_vs_interaction =",
            best_wi["lag"],
            "corr =",
            f"{best_wi['corr']:.6f}",
            "n =",
            best_wi["n"],
        )
        print(
            "best_lag weibo_vs_anti_ratio  =",
            best_wa["lag"],
            "corr =",
            f"{best_wa['corr']:.6f}",
            "n =",
            best_wa["n"],
        )
        print(
            "best_lag interaction_vs_anti_ratio =",
            best_ia["lag"],
            "corr =",
            f"{best_ia['corr']:.6f}",
            "n =",
            best_ia["n"],
        )


app = App(
    in_csv=Path("./outputs/step5/rumor_daily_merged.csv"),
    out_dir=Path("./outputs/step6"),
)
app.run()

## 7. Explain-Grouped Rumor Topic Word Clouds with Distinctive Term Analysis

In [ ]:
def find_font_path():
    local_dirs = [Path.cwd()]
    if "__file__" in globals():
        try:
            local_dirs.append(Path(__file__).resolve().parent)
        except Exception:
            pass

    for d in local_dirs:
        for name in ("SimHei.ttc", "SimHei.ttf", "SimHei.tt"):
            p = d / name
            if p.exists():
                return str(p)

    candidates = [
        "/System/Library/Fonts/PingFang.ttc",
        "/System/Library/Fonts/STHeiti Medium.ttc",
        "/System/Library/Fonts/STHeiti Light.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/System/Library/Fonts/Supplemental/Songti.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        "/Library/Fonts/Microsoft YaHei.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "C:\\Windows\\Fonts\\msyh.ttc",
        "C:\\Windows\\Fonts\\msyh.ttf",
        "C:\\Windows\\Fonts\\simhei.ttf",
        "C:\\Windows\\Fonts\\simsun.ttc",
    ]
    for p in candidates:
        if Path(p).exists():
            return p
    return None


def clean_text(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def build_tokenizer():
    try:
        import jieba

        try:
            jieba.setLogLevel(20)
        except Exception:
            pass

        def tokenize(text):
            return [w.strip() for w in jieba.lcut(text) if w.strip()]

        return tokenize
    except Exception:

        def tokenize(text):
            text = re.sub(r"\s+", "", str(text))
            if len(text) <= 1:
                return [text] if text else []
            return [text[i : i + 2] for i in range(len(text) - 1)]

        return tokenize


def build_stopwords():
    items = [
        "\u7684",
        "\u4e86",
        "\u662f",
        "\u6211",
        "\u4f60",
        "\u4ed6",
        "\u5979",
        "\u5b83",
        "\u4eec",
        "\u5728",
        "\u4e0e",
        "\u548c",
        "\u53ca",
        "\u6216",
        "\u4e5f",
        "\u90fd",
        "\u5c31",
        "\u800c",
        "\u4f46",
        "\u4e00\u4e2a",
        "\u6211\u4eec",
        "\u4f60\u4eec",
        "\u4ed6\u4eec",
        "\u8fd9\u79cd",
        "\u8fd9\u6837",
        "\u90a3\u4e2a",
        "\u8fd9\u4e2a",
        "\u8fd8",
        "\u5f88",
        "\u8fd9\u4e9b",
        "\u90a3\u4e9b",
        "\u6ca1\u6709",
        "\u56e0\u4e3a",
        "\u6240\u4ee5",
        "\u53ef\u4ee5",
        "\u76ee\u524d",
        "\u901a\u8fc7",
        "\u53ef\u80fd",
        "\u6709\u6548",
    ]
    return set(items)


def filter_tokens(tokens, stopwords):
    out = []
    for t in tokens:
        t = str(t).strip()
        if not t:
            continue
        if t in stopwords:
            continue
        if len(t) <= 1:
            continue
        if re.fullmatch(r"[\W_]+", t):
            continue
        out.append(t)
    return out


def label_id(label_text):
    h = hashlib.md5(str(label_text).encode("utf-8")).hexdigest()[:10]
    return f"label_{h}"


@dataclass(frozen=True)
class GroupStats:
    label_text: str
    group_id: str
    record_count: int
    token_count: int


class LogOdds:
    def __init__(self, alpha0=1000.0):
        self._alpha0 = float(alpha0)

    def compute_top(self, group_counts, all_counts, top_k):
        all_total = sum(all_counts.values())
        alpha0 = self._alpha0
        out = []
        group_total = sum(group_counts.values())
        rest_total = all_total - group_total

        for w, yw in group_counts.items():
            bg = all_counts.get(w, 0)
            if bg <= 0:
                continue
            alpha_w = alpha0 * (bg / all_total)
            y1 = float(yw)
            y2 = float(bg - yw)
            a = (y1 + alpha_w) / max(1e-12, (group_total - y1 + (alpha0 - alpha_w)))
            b = (y2 + alpha_w) / max(1e-12, (rest_total - y2 + (alpha0 - alpha_w)))
            lo = math.log(a) - math.log(b)
            var = 1.0 / max(1e-12, (y1 + alpha_w)) + 1.0 / max(1e-12, (y2 + alpha_w))
            z = lo / math.sqrt(var)
            out.append((w, int(yw), float(z)))

        out.sort(key=lambda x: x[2], reverse=True)
        return out[:top_k]


class WordCloudWriter:
    def __init__(self, font_path):
        self._font_path = font_path
        if not self._font_path:
            raise RuntimeError(
                "no font found for CJK rendering. set a valid font_path in find_font_path()"
            )
        self._WordCloud = WordCloud

    def write(self, freq, out_path, title=None):
        wc = self._WordCloud(
            width=1600,
            height=1000,
            background_color="white",
            font_path=self._font_path,
            max_words=200,
            collocations=False,
            prefer_horizontal=0.9,
            random_state=42,
        )
        wc.generate_from_frequencies(dict(freq))
        wc.to_file(str(out_path))


def run():
    in_path = Path("./data/CSDC-Rumor/fact.json")
    out_dir = Path("./outputs/step7/fact_wordclouds")
    out_dir.mkdir(parents=True, exist_ok=True)

    tokenize = build_tokenizer()
    stopwords = build_stopwords()
    font_path = find_font_path()

    group_counters = defaultdict(Counter)
    group_record_counts = Counter()
    all_counter = Counter()

    with in_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            label = obj.get("explain")
            if label is None:
                label = "unknown"
            gid = label_id(label)
            text = " ".join(
                [
                    clean_text(obj.get("title")),
                    clean_text(obj.get("rumor")),
                    clean_text(obj.get("abstract")),
                ]
            ).strip()
            if not text:
                continue
            toks = filter_tokens(tokenize(text), stopwords)
            if not toks:
                continue
            group_record_counts[gid] += 1
            group_counters[gid].update(toks)
            all_counter.update(toks)

    wcw = WordCloudWriter(font_path=font_path)
    wcw.write(all_counter, out_dir / "wordcloud_all.png")

    gid_to_label = {}
    with in_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            label = obj.get("explain")
            if label is None:
                label = "unknown"
            gid_to_label[label_id(label)] = str(label)

    stats = []
    for gid in sorted(group_counters.keys()):
        stats.append(
            GroupStats(
                label_text=gid_to_label.get(gid, "unknown"),
                group_id=gid,
                record_count=int(group_record_counts.get(gid, 0)),
                token_count=int(sum(group_counters[gid].values())),
            )
        )

    for s in stats:
        wcw.write(group_counters[s.group_id], out_dir / f"wordcloud_{s.group_id}.png")

    labels_csv = out_dir / "labels.csv"
    with labels_csv.open("w", encoding="utf-8", newline="") as f:
        f.write("group_id,label_text,record_count,token_count\n")
        for s in stats:
            f.write(
                f"{s.group_id},{json.dumps(s.label_text, ensure_ascii=False)},{s.record_count},{s.token_count}\n"
            )

    lo = LogOdds(alpha0=1000.0)
    distinct_csv = out_dir / "distinctive_terms.csv"
    with distinct_csv.open("w", encoding="utf-8", newline="") as f:
        f.write("group_id,term,term_count,z_score\n")
        for s in stats:
            top = lo.compute_top(group_counters[s.group_id], all_counter, top_k=40)
            for term, cnt, z in top:
                f.write(
                    f"{s.group_id},{json.dumps(term, ensure_ascii=False)},{cnt},{z:.4f}\n"
                )

    print("written =", str(out_dir / "wordcloud_all.png"))
    for s in stats:
        print(
            "written =",
            str(out_dir / f"wordcloud_{s.group_id}.png"),
            "records =",
            s.record_count,
            "tokens =",
            s.token_count,
        )
    print("written =", str(labels_csv))
    print("written =", str(distinct_csv))

    print("\nTop distinctive terms per group:")
    for s in stats:
        top = lo.compute_top(group_counters[s.group_id], all_counter, top_k=12)
        terms = [t[0] for t in top]
        print(s.group_id, json.dumps(s.label_text, ensure_ascii=False), terms)


run()

## 8. Dataset Profiling Pipeline for CSDC Rumor, News, and Legal Corpora

In [ ]:
class JsonIO:
    def load_json(self, path: Path):
        return json.loads(path.read_text(encoding="utf-8", errors="ignore"))

    def iter_jsonl(self, path: Path):
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                try:
                    yield json.loads(s)
                except Exception:
                    continue


class DateParser:
    def rumor_date_from_filename(self, name: str) -> str | None:
        m = re.match(r"^(\d{4}-\d{2}-\d{2})_", name)
        return m.group(1) if m else None

    def code_from_filename(self, name: str) -> str | None:
        if not name.endswith(".json"):
            return None
        base = name[:-5]
        if "_" not in base:
            return None
        return base.split("_", 1)[1].strip() or None

    def news_mmdd_from_filename(self, name: str) -> str | None:
        m = re.match(r"^(\d{2}-\d{2})\.(txt|json)$", name)
        return m.group(1) if m else None

    def mmdd_to_yyyy_mm_dd(self, mmdd: str, year: int = 2020) -> str | None:
        try:
            dt = datetime.strptime(f"{year}-{mmdd}", "%Y-%m-%d")
            return dt.strftime("%Y-%m-%d")
        except Exception:
            return None


@dataclass(frozen=True)
class ProfileSettings:
    rumor_root: Path
    news_root: Path
    legal_root: Path
    out_dir: Path
    year_for_news: int = 2020
    max_news_data_files: int | None = None
    max_news_comment_files: int | None = None


class RumorProfiler:
    def __init__(self, io: JsonIO, dp: DateParser):
        self._io = io
        self._dp = dp

    def run(self, rumor_root: Path) -> dict:
        weibo_dir = rumor_root / "rumor_weibo"
        fc_dir = rumor_root / "rumor_forward_comment"
        fact_path = rumor_root / "fact.json"

        weibo_files = sorted(weibo_dir.glob("*.json"))
        fc_files = sorted(fc_dir.glob("*.json"))

        weibo_codes = set()
        fc_codes = set()

        result_counter = Counter()
        visit_stats = []
        publish_dates = Counter()

        for p in weibo_files:
            code = self._dp.code_from_filename(p.name)
            if code:
                weibo_codes.add(code)
            d = self._dp.rumor_date_from_filename(p.name)
            if d:
                publish_dates[d] += 1
            obj = self._safe_load_obj(p)
            if not isinstance(obj, dict):
                continue
            res = str(obj.get("result") or "").strip()
            if res:
                result_counter[res] += 1
            vt = obj.get("visitTimes")
            try:
                if vt is not None and str(vt).strip() != "":
                    visit_stats.append(int(str(vt).strip()))
            except Exception:
                pass

        kind_counter = Counter()
        fc_dates = Counter()
        fc_items_total = 0
        fc_text_nonempty = 0

        for p in fc_files:
            code = self._dp.code_from_filename(p.name)
            if code:
                fc_codes.add(code)
            d = self._dp.rumor_date_from_filename(p.name)
            if d:
                fc_dates[d] += 1
            arr = self._safe_load_obj(p)
            if not isinstance(arr, list):
                continue
            for item in arr:
                if not isinstance(item, dict):
                    continue
                k = str(item.get("comment_or_forward") or "").strip()
                if k:
                    kind_counter[k] += 1
                fc_items_total += 1
                txt = item.get("text")
                if txt is not None and str(txt).strip():
                    fc_text_nonempty += 1

        missing_fc_for_weibo = sorted(list(weibo_codes - fc_codes))
        missing_weibo_for_fc = sorted(list(fc_codes - weibo_codes))

        fact_explain_counter = Counter()
        fact_tag_counter = Counter()
        fact_total = 0

        if fact_path.exists():
            for obj in self._io.iter_jsonl(fact_path):
                if not isinstance(obj, dict):
                    continue
                fact_total += 1
                explain = str(obj.get("explain") or "").strip() or "unknown"
                tag = str(obj.get("tag") or "").strip() or "unknown"
                fact_explain_counter[explain] += 1
                fact_tag_counter[tag] += 1

        out = {
            "rumor_weibo": {
                "files": len(weibo_files),
                "unique_codes": len(weibo_codes),
                "result_top": result_counter.most_common(20),
                "visit_times": self._basic_stats(visit_stats),
                "date_counts_top": publish_dates.most_common(20),
            },
            "rumor_forward_comment": {
                "files": len(fc_files),
                "unique_codes": len(fc_codes),
                "items_total": fc_items_total,
                "text_nonempty": fc_text_nonempty,
                "kind_counts": dict(kind_counter),
                "date_counts_top": fc_dates.most_common(20),
            },
            "pairing": {
                "weibo_without_forward_comment_count": len(missing_fc_for_weibo),
                "forward_comment_without_weibo_count": len(missing_weibo_for_fc),
                "weibo_without_forward_comment_sample": missing_fc_for_weibo[:20],
                "forward_comment_without_weibo_sample": missing_weibo_for_fc[:20],
            },
            "fact": {
                "records": fact_total,
                "explain_top": fact_explain_counter.most_common(30),
                "tag_top": fact_tag_counter.most_common(30),
            },
        }
        return out

    def _safe_load_obj(self, path: Path):
        try:
            return self._io.load_json(path)
        except Exception:
            return None

    def _basic_stats(self, xs: list[int]) -> dict:
        if not xs:
            return {"n": 0}
        x = sorted(xs)
        n = len(x)

        def pick(p: float) -> int:
            idx = int(round(p * (n - 1)))
            idx = max(0, min(n - 1, idx))
            return int(x[idx])

        return {
            "n": n,
            "min": int(x[0]),
            "p25": pick(0.25),
            "p50": pick(0.50),
            "p75": pick(0.75),
            "p90": pick(0.90),
            "max": int(x[-1]),
        }


class NewsProfiler:
    def __init__(self, io: JsonIO, dp: DateParser):
        self._io = io
        self._dp = dp

    def run(
        self,
        news_root: Path,
        year: int,
        max_data_files: int | None,
        max_comment_files: int | None,
    ) -> dict:
        data_dir = news_root / "data"
        comment_dir = news_root / "comment"

        data_files = sorted(
            list(data_dir.glob("*.txt")) + list(data_dir.glob("*.json"))
        )
        comment_files = sorted(
            list(comment_dir.glob("*.txt")) + list(comment_dir.glob("*.json"))
        )

        if max_data_files is not None:
            data_files = data_files[: max(0, int(max_data_files))]
        if max_comment_files is not None:
            comment_files = comment_files[: max(0, int(max_comment_files))]

        data_counts_by_day = Counter()
        type_counter = Counter()
        keyword_counter = Counter()
        data_items_total = 0
        data_missing_meta = 0

        for p in data_files:
            mmdd = self._dp.news_mmdd_from_filename(p.name)
            day = self._dp.mmdd_to_yyyy_mm_dd(mmdd, year=year) if mmdd else "unknown"
            obj = self._safe_load_obj(p)
            if not isinstance(obj, list):
                continue
            data_counts_by_day[day] += len(obj)
            data_items_total += len(obj)
            for item in obj:
                if not isinstance(item, dict):
                    continue
                meta = item.get("meta")
                if not isinstance(meta, dict):
                    data_missing_meta += 1
                    continue
                t = str(meta.get("type") or "").strip()
                if t:
                    type_counter[t] += 1
                kw = meta.get("keyword")
                if isinstance(kw, list):
                    for k in kw:
                        s = str(k).strip()
                        if s:
                            keyword_counter[s] += 1
                elif kw is not None:
                    s = str(kw).strip()
                    if s:
                        keyword_counter[s] += 1

        comment_counts_by_day = Counter()
        area_counter = Counter()
        comment_items_total = 0
        comment_entries_total = 0

        for p in comment_files:
            mmdd = self._dp.news_mmdd_from_filename(p.name)
            day = self._dp.mmdd_to_yyyy_mm_dd(mmdd, year=year) if mmdd else "unknown"
            obj = self._safe_load_obj(p)
            if not isinstance(obj, list):
                continue
            comment_items_total += len(obj)
            for item in obj:
                if not isinstance(item, dict):
                    continue
                arr = item.get("comment")
                if not isinstance(arr, list):
                    continue
                comment_counts_by_day[day] += len(arr)
                comment_entries_total += len(arr)
                for c in arr:
                    if not isinstance(c, dict):
                        continue
                    area = str(c.get("area") or "").strip() or "unknown"
                    area_counter[area] += 1

        out = {
            "news_data": {
                "files_scanned": len(data_files),
                "items_total": data_items_total,
                "missing_meta": data_missing_meta,
                "counts_by_day_top": data_counts_by_day.most_common(20),
                "type_top": type_counter.most_common(30),
                "keyword_top": keyword_counter.most_common(50),
            },
            "news_comment": {
                "files_scanned": len(comment_files),
                "news_items_with_comments_total": comment_items_total,
                "comment_entries_total": comment_entries_total,
                "counts_by_day_top": comment_counts_by_day.most_common(20),
                "area_top": area_counter.most_common(30),
            },
        }
        return out

    def _safe_load_obj(self, path: Path):
        try:
            return self._io.load_json(path)
        except Exception:
            return None


class LegalProfiler:
    def __init__(self, io: JsonIO):
        self._io = io

    def run(self, legal_root: Path) -> dict:
        candidates = [
            legal_root / "legal_doc.json",
            legal_root / "CSDC-Legal.json",
            legal_root / "CSDC-Legal" / "legal_doc.json",
        ]
        path = None
        for c in candidates:
            if c.exists():
                path = c
                break
        if not path:
            return {
                "error": "legal_doc.json not found",
                "candidates": [str(x) for x in candidates],
            }

        obj = self._safe_load_obj(path)
        if not isinstance(obj, list):
            return {"error": "legal_doc.json is not a json list", "path": str(path)}

        n = 0
        title_len = []
        content_len = []
        missing_title = 0
        missing_content = 0

        for item in obj:
            if not isinstance(item, dict):
                continue
            n += 1
            title = item.get("title")
            content = item.get("content")
            if title is None or str(title).strip() == "":
                missing_title += 1
            else:
                title_len.append(len(str(title)))
            if content is None or str(content).strip() == "":
                missing_content += 1
            else:
                content_len.append(len(str(content)))

        out = {
            "path": str(path),
            "records": n,
            "missing_title": missing_title,
            "missing_content": missing_content,
            "title_length": self._basic_stats(title_len),
            "content_length": self._basic_stats(content_len),
        }
        return out

    def _safe_load_obj(self, path: Path):
        try:
            return self._io.load_json(path)
        except Exception:
            return None

    def _basic_stats(self, xs: list[int]) -> dict:
        if not xs:
            return {"n": 0}
        x = sorted(xs)
        n = len(x)

        def pick(p: float) -> int:
            idx = int(round(p * (n - 1)))
            idx = max(0, min(n - 1, idx))
            return int(x[idx])

        return {
            "n": n,
            "min": int(x[0]),
            "p25": pick(0.25),
            "p50": pick(0.50),
            "p75": pick(0.75),
            "p90": pick(0.90),
            "max": int(x[-1]),
        }


class ProfileWriter:
    def write_json(self, path: Path, obj: dict) -> None:
        path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


class DatasetProfileApp:
    def __init__(self, settings: ProfileSettings):
        self._settings = settings
        self._io = JsonIO()
        self._dp = DateParser()
        self._rumor = RumorProfiler(self._io, self._dp)
        self._news = NewsProfiler(self._io, self._dp)
        self._legal = LegalProfiler(self._io)
        self._writer = ProfileWriter()

    def run(self) -> dict:
        s = self._settings
        s.out_dir.mkdir(parents=True, exist_ok=True)

        rumor_profile = self._rumor.run(s.rumor_root)
        news_profile = self._news.run(
            s.news_root,
            s.year_for_news,
            s.max_news_data_files,
            s.max_news_comment_files,
        )
        legal_profile = self._legal.run(s.legal_root)

        summary = {
            "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "paths": {
                "rumor_root": str(s.rumor_root),
                "news_root": str(s.news_root),
                "legal_root": str(s.legal_root),
            },
            "rumor": {
                "weibo_files": rumor_profile.get("rumor_weibo", {}).get("files"),
                "fc_files": rumor_profile.get("rumor_forward_comment", {}).get("files"),
                "fact_records": rumor_profile.get("fact", {}).get("records"),
                "weibo_without_fc": rumor_profile.get("pairing", {}).get(
                    "weibo_without_forward_comment_count"
                ),
            },
            "news": {
                "data_files_scanned": news_profile.get("news_data", {}).get(
                    "files_scanned"
                ),
                "comment_files_scanned": news_profile.get("news_comment", {}).get(
                    "files_scanned"
                ),
            },
            "legal": {
                "records": legal_profile.get("records"),
            },
        }

        self._writer.write_json(s.out_dir / "profile_summary.json", summary)
        self._writer.write_json(s.out_dir / "rumor_profile.json", rumor_profile)
        self._writer.write_json(s.out_dir / "news_profile.json", news_profile)
        self._writer.write_json(s.out_dir / "legal_profile.json", legal_profile)

        print("written =", str(s.out_dir / "profile_summary.json"))
        print("written =", str(s.out_dir / "rumor_profile.json"))
        print("written =", str(s.out_dir / "news_profile.json"))
        print("written =", str(s.out_dir / "legal_profile.json"))

        return summary


settings = ProfileSettings(
    rumor_root=Path("./data/CSDC-Rumor"),
    news_root=Path("./data/CSDC-News"),
    legal_root=Path("./data/CSDC-Legal"),
    out_dir=Path("./outputs/step8"),
    year_for_news=2020,
    max_news_data_files=None,
    max_news_comment_files=None,
)

app = DatasetProfileApp(settings)
summary = app.run()
summary

## 9. Weakly Supervised Sentiment Modeling and Trends

In [ ]:
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


def clean_text(s):
    if not s:
        return ""
    s = re.sub(r"https?://\S+", " ", str(s))
    s = re.sub(r"@\S+", " ", s)
    s = re.sub(r"#\S+#", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def normalize(s):
    s = re.sub(r"[0-9]+", "0", str(s))
    return re.sub(r"\s+", " ", s).strip()


@dataclass(frozen=True)
class Record:
    day: date
    text: str
    y: str


class WeakLabeler:
    def __init__(self):
        self.pos = ["太好了", "真好", "很好", "加油", "希望", "致敬", "感谢"]
        self.neg = [
            "太可怕了",
            "好可怕",
            "恶心",
            "无语",
            "垃圾",
            "骗子",
            "造谣",
            "谣言",
        ]

    def label(self, t):
        p = sum(1 for w in self.pos if w in t)
        n = sum(1 for w in self.neg if w in t)
        if p >= 2 and n == 0:
            return "pos"
        if n >= 2 and p == 0:
            return "neg"
        return None

    def mask(self, t):
        for w in self.pos + self.neg:
            t = t.replace(w, " ")
        return re.sub(r"\s+", " ", t).strip()


def build_dataset(comment_dir: Path, seed=42):
    labeler = WeakLabeler()
    pos, neg = [], []
    stats = Counter()

    for p in sorted(comment_dir.glob("*.txt")):
        stem = p.stem
        if not re.fullmatch(r"\d{2}-\d{2}", stem):
            continue
        mm, dd = map(int, stem.split("-"))
        try:
            day = date(2020, mm, dd)
        except Exception:
            continue

        data = json.loads(p.read_text(encoding="utf-8"))
        for item in data:
            for c in item.get("comment", []):
                text = normalize(clean_text(c.get("content")))
                if len(text) < 8:
                    stats["short"] += 1
                    continue
                y = labeler.label(text)
                if not y:
                    stats["unlabeled"] += 1
                    continue
                masked = labeler.mask(text)
                rec = Record(day, masked, y)
                (pos if y == "pos" else neg).append(rec)

    m = min(len(pos), len(neg))
    random.Random(seed).shuffle(pos)
    random.Random(seed).shuffle(neg)
    all_rec = pos[:m] + neg[:m]
    random.shuffle(all_rec)

    return all_rec, stats


def train_tfidf(records, out_dir, analyzer="char"):
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.metrics import classification_report
    import joblib

    X = [r.text for r in records]
    y = [r.y for r in records]
    g = [r.day for r in records]

    tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X, y, g))
    Xtr, Xte = [X[i] for i in tr], [X[i] for i in te]
    ytr, yte = [y[i] for i in tr], [y[i] for i in te]

    vec = TfidfVectorizer(
        analyzer=analyzer,
        ngram_range=(2, 5) if analyzer == "char" else (1, 2),
        min_df=5,
        max_df=0.95,
    )
    Xtr = vec.fit_transform(Xtr)
    Xte = vec.transform(Xte)

    clf = (
        LogisticRegression(max_iter=3000, class_weight="balanced")
        if analyzer == "char"
        else CalibratedClassifierCV(LinearSVC(class_weight="balanced"), cv=3)
    )
    clf.fit(Xtr, ytr)

    rep = classification_report(yte, clf.predict(Xte), output_dict=True)

    ensure_dir(out_dir)
    joblib.dump({"vec": vec, "clf": clf}, out_dir / "model.joblib")
    (out_dir / "eval.json").write_text(json.dumps(rep, indent=2, ensure_ascii=False))

    return rep


def train_bert(records, bert_dir, out_dir):
    import torch
    from transformers import AutoTokenizer, AutoModel
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.svm import LinearSVC
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.metrics import classification_report
    import joblib

    tokenizer = AutoTokenizer.from_pretrained(
        bert_dir, use_fast=False, local_files_only=True
    )
    model = AutoModel.from_pretrained(
        bert_dir, local_files_only=True, weights_only=False
    ).eval()

    def embed(texts):
        with torch.no_grad():
            enc = tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            )
            out = model(**enc)
            return out.last_hidden_state.mean(dim=1).numpy()

    X = [r.text for r in records]
    y = [r.y for r in records]
    g = [r.day for r in records]

    tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(X, y, g))
    Xtr, Xte = embed([X[i] for i in tr]), embed([X[i] for i in te])
    ytr, yte = [y[i] for i in tr], [y[i] for i in te]

    clf = CalibratedClassifierCV(LinearSVC(class_weight="balanced"), cv=3)
    clf.fit(Xtr, ytr)

    rep = classification_report(yte, clf.predict(Xte), output_dict=True)

    ensure_dir(out_dir)
    joblib.dump(clf, out_dir / "svm.joblib")
    (out_dir / "eval.json").write_text(json.dumps(rep, indent=2, ensure_ascii=False))

    return rep


OUT = Path("./outputs/step9")
DATA = Path("./data/CSDC-News/comment")
BERT = Path("./chinese-bert-wwm-ext")

records, stats = build_dataset(DATA)
print("records =", len(records), "stats =", stats)

rep1 = train_tfidf(records, OUT / "tfidf_char_lr", "char")
rep2 = train_tfidf(records, OUT / "tfidf_word_svm", "word")
rep3 = train_bert(records, BERT, OUT / "bert_svm")

summary = {
    "tfidf_char_lr": rep1["macro avg"]["f1-score"],
    "tfidf_word_svm": rep2["macro avg"]["f1-score"],
    "bert_svm": rep3["macro avg"]["f1-score"],
}

(OUT / "model_comparison.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False)
)
print("Step9 finished:", summary)

## 10. Rumor Diffusion Feature Analysis with Sentiment Integration

In [ ]:
def code_from_filename(name: str):
    if "_" not in name:
        return None
    return name.split("_", 1)[1].replace(".json", "")


def day_from_filename(name: str):
    m = re.match(r"^(\d{4}-\d{2}-\d{2})_", name)
    if not m:
        return None
    try:
        return datetime.strptime(m.group(1), "%Y-%m-%d").date()
    except Exception:
        return None


def parse_date_any(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        try:
            return datetime.fromtimestamp(float(x)).date()
        except Exception:
            return None
    s = str(x).strip().replace("T", " ").replace("Z", "")
    for fmt in (
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%Y-%m-%d %H:%M",
        "%Y/%m/%d %H:%M",
        "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M:%S",
    ):
        try:
            return datetime.strptime(s[: len(fmt)], fmt).date()
        except Exception:
            pass
    return None


def clean_text(s):
    if s is None:
        return ""
    s = re.sub(r"https?://\S+", " ", str(s))
    return re.sub(r"\s+", " ", s).strip()


def is_repost_stub(t: str):
    if not t:
        return True
    t = re.sub(r"^\S+[:：]\s*", "", t).strip()
    return t in {"转发微博", "Repost", "转发", "RT"}


@dataclass
class RumorFeat:
    code: str
    publish_day: str
    forward_count: int
    comment_count: int
    total_interactions: int
    forward_ratio: float
    active_days: int
    peak_day: str
    peak_count: int
    valid_texts: int
    neg_ratio: float
    peak_day_neg_ratio: float


RUMOR_ROOT = Path("./data/CSDC-Rumor")
OUT_DIR = Path("./outputs/step10")
MODEL_DIR = Path("./chinese-bert-wwm-ext")
SVM_PATH = Path("./outputs/step9/bert_svm/svm.joblib")

ensure_dir(OUT_DIR)

device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR, use_fast=False, local_files_only=True
)
bert = AutoModel.from_pretrained(
    MODEL_DIR, local_files_only=True, weights_only=False
).eval()
svm = joblib.load(SVM_PATH)


def embed_batched(texts, batch_size=16):
    if not texts:
        return np.zeros((0, bert.config.hidden_size), dtype="float32")
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        with torch.no_grad():
            enc = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            )
            out = bert(**enc)
            x = out.last_hidden_state.mean(dim=1).numpy()
            outs.append(x)
    return np.vstack(outs)


weibo_by_code = {}
for p in (RUMOR_ROOT / "rumor_weibo").glob("*.json"):
    try:
        obj = json.loads(p.read_text(encoding="utf-8", errors="ignore"))
    except Exception:
        continue
    code = code_from_filename(p.name)
    if code:
        weibo_by_code[code] = obj


feats = []
peak_samples = []
totals = []
ratios = []

for p in (RUMOR_ROOT / "rumor_forward_comment").glob("*.json"):
    try:
        arr = json.loads(p.read_text(encoding="utf-8", errors="ignore"))
    except Exception:
        continue
    if not isinstance(arr, list):
        continue

    code = code_from_filename(p.name)
    if not code:
        continue

    forward_count = 0
    comment_count = 0
    day_counter = Counter()
    texts = []
    text_days = []

    w = weibo_by_code.get(code, {})
    publish_day = parse_date_any(w.get("publishTime")) or day_from_filename(p.name)

    for it in arr:
        k = it.get("comment_or_forward")
        if k == "forward":
            forward_count += 1
        elif k == "comment":
            comment_count += 1

        d = parse_date_any(it.get("date")) or publish_day
        if d:
            day_counter[d] += 1

        txt = clean_text(it.get("text"))
        if txt and not is_repost_stub(txt):
            texts.append(txt)
            text_days.append(d)

    total = forward_count + comment_count
    if total == 0 or not day_counter:
        continue

    days = sorted(day_counter.keys())
    first_day = days[0]
    last_day = days[-1]
    peak_day, peak_count = max(day_counter.items(), key=lambda x: x[1])
    active_days = (last_day - first_day).days + 1
    forward_ratio = forward_count / total

    X = embed_batched(texts, batch_size=16)
    if len(X):
        prob = svm.predict_proba(X)
        idx = list(svm.classes_).index("pos")
        neg_flags = prob[:, idx] < 0.5
        neg_ratio = float(neg_flags.mean())
        peak_mask = [d == peak_day for d in text_days]
        peak_day_neg_ratio = (
            float(neg_flags[np.array(peak_mask)].mean()) if any(peak_mask) else 0.0
        )
    else:
        neg_ratio = 0.0
        peak_day_neg_ratio = 0.0

    feats.append(
        RumorFeat(
            code=code,
            publish_day=publish_day.isoformat() if publish_day else "",
            forward_count=forward_count,
            comment_count=comment_count,
            total_interactions=total,
            forward_ratio=forward_ratio,
            active_days=active_days,
            peak_day=peak_day.isoformat(),
            peak_count=peak_count,
            valid_texts=len(texts),
            neg_ratio=neg_ratio,
            peak_day_neg_ratio=peak_day_neg_ratio,
        )
    )

    totals.append(total)
    ratios.append(forward_ratio)

    for t, d in zip(texts, text_days):
        if d == peak_day and len(peak_samples) < 300:
            peak_samples.append(
                {"code": code, "peak_day": peak_day.isoformat(), "text": t[:200]}
            )


with (OUT_DIR / "rumor_code_features.csv").open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=RumorFeat.__dataclass_fields__.keys())
    w.writeheader()
    for r in feats:
        w.writerow(r.__dict__)

with (OUT_DIR / "peak_day_samples.csv").open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["code", "peak_day", "text"])
    w.writeheader()
    for r in peak_samples:
        w.writerow(r)

plt.figure(figsize=(8, 6))
plt.loglog(sorted(totals, reverse=True))
plt.tight_layout()
plt.savefig(OUT_DIR / "interaction_longtail_rank.png", dpi=300)
plt.close()

plt.figure(figsize=(8, 6))
plt.hist(ratios, bins=30)
plt.tight_layout()
plt.savefig(OUT_DIR / "forward_ratio_hist.png", dpi=300)
plt.close()

x = np.log1p(np.array(totals))
y = np.array([r.neg_ratio for r in feats])
plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.5)
plt.xlabel("log1p(total_interactions)")
plt.ylabel("neg_ratio")
plt.tight_layout()
plt.savefig(OUT_DIR / "neg_ratio_vs_total.png", dpi=300)
plt.close()

with (OUT_DIR / "top_rumors_by_total_interactions.csv").open(
    "w", encoding="utf-8", newline=""
) as f:
    w = csv.writer(f)
    w.writerow(
        [
            "rank",
            "code",
            "total_interactions",
            "forward_count",
            "comment_count",
            "forward_ratio",
            "peak_day",
            "peak_count",
        ]
    )
    for i, r in enumerate(
        sorted(feats, key=lambda x: x.total_interactions, reverse=True)[:200]
    ):
        w.writerow(
            [
                i + 1,
                r.code,
                r.total_interactions,
                r.forward_count,
                r.comment_count,
                f"{r.forward_ratio:.6f}",
                r.peak_day,
                r.peak_count,
            ]
        )


out_dir = Path("./outputs/step10")
src = out_dir / "rumor_code_features.csv"
dst = out_dir / "top_rumors_by_total_interactions.csv"

rows = []
with src.open("r", encoding="utf-8", newline="") as f:
    r = csv.DictReader(f)
    for row in r:
        try:
            row["_total"] = int(float(row.get("total_interactions", 0) or 0))
        except Exception:
            row["_total"] = 0
        rows.append(row)

rows.sort(key=lambda x: x["_total"], reverse=True)

with dst.open("w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(
        [
            "rank",
            "code",
            "total_interactions",
            "forward_count",
            "comment_count",
            "forward_ratio",
            "publish_day",
            "peak_day",
            "peak_count",
            "title",
        ]
    )
    for i, row in enumerate(rows[:200]):
        title = row.get("title", "")
        w.writerow(
            [
                i + 1,
                row.get("code", ""),
                row.get("total_interactions", ""),
                row.get("forward_count", ""),
                row.get("comment_count", ""),
                row.get("forward_ratio", ""),
                row.get("publish_day", ""),
                row.get("peak_day", ""),
                row.get("peak_count", ""),
                title,
            ]
        )

print("written =", str(dst))
print("Step10 finished")